# Práctica 10 — Segmentación de Viajes de Taxi (NYC)
**Machine Learning · ISTER 2026 · Ing. David Minango, PhD**

---
**Objetivo:** Aplicar el flujo completo de un proyecto de clustering a los viajes de taxi de Nueva York: preparar y escalar, elegir K, segmentar con K-means, **evaluar con las tres métricas internas** (Silhouette, Davies-Bouldin, Calinski-Harabasz), **perfilar y bautizar** los segmentos, y detectar **viajes anómalos** con DBScan.

**Contexto:** Eres el analista de datos de una empresa de taxis. Te piden descubrir qué tipos de viaje existen (para diseñar tarifas y promociones) y detectar viajes anómalos (posibles errores de taxímetro o fraudes). Nadie te dio etiquetas: es un problema 100% no supervisado.

> ⏱️ Duración estimada: ~60 minutos
> 🔧 Completa las celdas marcadas con **TU CÓDIGO**. Las demás solo ejecútalas.
> 💾 Guarda una copia en tu Drive: `Archivo → Guardar una copia en Drive`

## Parte 0 — Setup (Solo ejecutar)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster       import KMeans, DBSCAN
from sklearn.neighbors     import NearestNeighbors
from sklearn.metrics       import silhouette_score, davies_bouldin_score, calinski_harabasz_score

print("✅ Librerías cargadas")

## Parte 1 — Preparar y Escalar

El dataset **taxis** de seaborn contiene más de 6.000 viajes reales de taxi en NYC (marzo 2019). Usaremos las variables numéricas del viaje: distancia, tarifa, propina y costo total.

In [ ]:
# Celda 1.1 — Dado: cargar y limpiar los viajes
features = ['distance', 'fare', 'tip', 'total']

df = sns.load_dataset('taxis')
df = df.dropna(subset=features)                    # quitamos viajes con datos faltantes
df = df[df['distance'] > 0]                        # quitamos viajes de distancia 0
df = df.sample(1500, random_state=42).reset_index(drop=True)

print(f"Viajes en la muestra: {df.shape[0]}")
print(df[features].describe().round(2))

In [ ]:
# Celda 1.2 — 🔧 TU CÓDIGO: escalar las features
# distance va de 0 a ~37 millas y total de ~$4 a ~$175:
# escalas distintas → StandardScaler obligatorio.

X = df[features].copy()

scaler   = ___________          # StandardScaler
X_scaled = ___________          # fit_transform sobre X

print("Forma de X_scaled:", X_scaled.shape)
print("Media (~0):", X_scaled.mean(axis=0).round(2))
print("Desv. (~1):", X_scaled.std(axis=0).round(2))

### ❓ Preguntas — Antes de modelar
1. ¿Qué pasaría con las distancias si NO escalaras (compara los rangos de `distance` y `total`)?
2. ¿Qué tipos de viaje esperas encontrar (pista: viajes cortos de ciudad vs viajes largos al aeropuerto)?

_Responde aquí:_

## Parte 2 — Elegir K y Segmentar

In [ ]:
# Celda 2.1 — 🔧 TU CÓDIGO: codo + silhouette para k de 2 a 8
# Para cada k: ajusta KMeans, guarda inercia y silhouette

print(f"{'k':>3} {'inercia':>12} {'silhouette':>12}")
for k in range(2, 9):
    km  = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    lab = ___________          # fit_predict sobre X_scaled
    sil = ___________          # silhouette_score(X_scaled, lab)
    print(f"{k:>3} {km.inertia_:>12.1f} {sil:>12.3f}")

In [ ]:
# Celda 2.2 — 🔧 TU CÓDIGO: K-means final y visualización
# Elige tu mejor k y ajusta el modelo final.
# Guarda las etiquetas en df['cluster_km'] y grafica distance vs total.

mejor_k = ___________          # tu elección según la celda anterior
km = KMeans(n_clusters=mejor_k, init='k-means++', n_init=10, random_state=42)
df['cluster_km'] = ___________          # fit_predict

plt.figure(figsize=(7.5, 5))
plt.scatter(df['distance'], df['total'], c=df['cluster_km'],
            cmap='viridis', s=20, edgecolor='k', linewidth=0.2)
plt.xlabel('Distancia (millas)')
plt.ylabel('Costo total (USD)')
plt.title(f'K-means (k={mejor_k}) — Segmentos de viajes de taxi')
plt.colorbar(label='Clúster')
plt.tight_layout()
plt.show()

### ❓ Preguntas — Sobre el K
1. ¿Qué k elegiste? ¿El codo y el silhouette apuntaban al mismo valor?
2. Mirando el scatter, ¿los grupos que ves tienen sentido para una empresa de taxis?

_Responde aquí:_

## Parte 3 — Evaluar con las Tres Métricas

En clustering no hay accuracy: medimos la **geometría** de los grupos.

| Métrica | Qué mide | Mejor |
|---|---|---|
| Silhouette | cohesión vs separación por punto | alto ↑ |
| Davies-Bouldin | dispersión ÷ distancia entre centroides | bajo ↓ |
| Calinski-Harabasz | separación entre ÷ dispersión dentro | alto ↑ |

In [ ]:
# Celda 3.1 — 🔧 TU CÓDIGO: las tres métricas de tu modelo final

labels = df['cluster_km']
sil = ___________          # silhouette_score(X_scaled, labels)
db  = ___________          # davies_bouldin_score(X_scaled, labels)
ch  = ___________          # calinski_harabasz_score(X_scaled, labels)

print(f"Silhouette        : {sil:.3f}   (alto = mejor)")
print(f"Davies-Bouldin    : {db:.3f}   (bajo = mejor)")
print(f"Calinski-Harabasz : {ch:.1f}   (alto = mejor)")

### ❓ Preguntas — Sobre la calidad
1. Según la escala vista en clase (≥0.5 fuerte · 0.25–0.5 razonable · <0.25 débil), ¿qué indica tu silhouette?
2. ¿Por qué en clustering necesitamos estas métricas en lugar de accuracy?

_Responde aquí:_

## Parte 4 — Perfilar y Bautizar los Segmentos

El número de clúster (0, 1, 2…) es arbitrario. Lo que comunica es el **perfil**: el promedio de cada variable por grupo, comparado con el promedio global.

In [ ]:
# Celda 4.1 — 🔧 TU CÓDIGO: perfil de cada clúster
# 1. Promedio de las features por clúster (groupby + mean)
# 2. Cuántos viajes tiene cada clúster
# 3. Compara contra el promedio global

perfil = ___________          # df.groupby('cluster_km')[features].mean().round(2)
perfil['n_viajes'] = df['cluster_km'].value_counts().sort_index()
print(perfil)

print("\nPromedio global:")
print(df[features].mean().round(2))

In [ ]:
# Celda 4.2 — 🔧 TU CÓDIGO (en comentarios): nombra cada segmento
# Lee la tabla de perfiles y bautiza cada clúster con un nombre
# que el gerente de la empresa de taxis entendería. Ejemplo:
#
# Clúster 0: "________________"  → porque ________________
# Clúster 1: "________________"  → porque ________________
# Clúster 2: "________________"  → porque ________________
#
# Pistas: ¿viaje corto de ciudad? ¿viaje largo (aeropuerto)?
#         ¿cliente que da buenas propinas?

### ❓ Preguntas — Sobre los segmentos
1. ¿Qué acción comercial propondrías para cada segmento (promoción, tarifa plana, fidelización)?
2. ¿Cuál de tus segmentos es el más valioso para la empresa? ¿Por qué?

_Responde aquí:_

## Parte 5 — Viajes Anómalos con DBScan

K-means asigna **todos** los viajes a un grupo. DBScan, en cambio, puede decir "este viaje no encaja en ningún patrón" (etiqueta −1): perfecto para auditar taxímetros.

In [ ]:
# Celda 5.1 — Dado: gráfico de k-distancias
min_samples = 5                     # regla: features + 1
nn = NearestNeighbors(n_neighbors=min_samples).fit(X_scaled)
dist, _ = nn.kneighbors(X_scaled)
k_dist = np.sort(dist[:, -1])

plt.figure(figsize=(8, 4.5))
plt.plot(k_dist, color='#3b82f6', lw=2)
plt.xlabel('Puntos ordenados')
plt.ylabel(f'Distancia al vecino {min_samples}')
plt.title('k-distancias — el codo sugiere el eps')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Celda 5.2 — 🔧 TU CÓDIGO: DBScan y visualización de anomalías
# 1. Aplica DBSCAN con el eps del codo (p.ej. 0.4) y min_samples=5
# 2. Guarda etiquetas en df['cluster_db'] y cuenta outliers
# 3. Grafica distance vs total: normales en color, outliers en X roja

db = ___________               # DBSCAN(eps=..., min_samples=5)
df['cluster_db'] = ___________ # fit_predict sobre X_scaled
es_outlier = df['cluster_db'] == -1

print(f"Viajes anómalos detectados: {es_outlier.sum()} de {len(df)}")

plt.figure(figsize=(7.5, 5))
# ___ scatter de los NO anómalos (df[~es_outlier]) coloreado por cluster_db ___
# ___ scatter de los anómalos (df[es_outlier]) en rojo con marker='x' ___
plt.xlabel('Distancia (millas)')
plt.ylabel('Costo total (USD)')
plt.title('DBScan — viajes anómalos marcados en rojo')
plt.legend()
plt.tight_layout()
plt.show()

# Inspecciona los viajes anómalos: ¿tarifas enormes? ¿propinas rarísimas?
print(df[es_outlier][features].sort_values('total', ascending=False).head(10))

### ❓ Preguntas — Sobre las anomalías
1. ¿Qué tienen de raro los viajes marcados (mira distancia vs costo)?
2. ¿Cuáles investigarías primero si fueras el auditor: costo altísimo con distancia corta, o distancia enorme con costo normal? ¿Por qué?

_Responde aquí:_

## ✅ Entrega

1. **Notebook completo:** todas las celdas 🔧 TU CÓDIGO resueltas y ejecutadas, preguntas ❓ respondidas en celdas Markdown. Exportar como `.ipynb`.
2. **Conclusión integradora:** resume tu proyecto en un párrafo como si se lo contaras al gerente: cuántos tipos de viaje encontraste, cómo verificaste que los grupos son de calidad (menciona al menos una métrica y qué indica), y qué recomendarías hacer con los viajes anómalos detectados.

_Escribe tu conclusión aquí:_